# Adversarial consistency with a perturbation curriculum

CISC7404-002 · Coursework 1 · MC647418 · University of Macau

This notebook contains the complete training and official evaluation implementation. The compact submission opens in record-review mode: it checks the saved run records, displays the frozen five-seed results and uses the supplied figure files. Set `FRESH_TRAIN = True` in the first code cell to download Fashion-MNIST and retrain all thirty confirmation trajectories. Model weights and raw image files are regenerated when needed.


## 1. Environment, data and fixed network

Only the original Fashion-MNIST training images enter optimization. A fixed stratified split reserves 500 examples per class for validation. The test set is used after the configuration is frozen.

In [1]:
"""Shared data, fixed architecture, official PGD and correctness checks."""
from __future__ import annotations

import argparse
import csv
import gzip
import hashlib
import json
import math
import os
import platform
import random
import struct
import time
import urllib.request
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import torch
from torch import nn
from torch.nn import functional as F

ROOT = Path.cwd()
FILES = {
    "train-images-idx3-ubyte.gz": "8d4fb7e6c68d591d4c3dfef9ec88bf0d",
    "train-labels-idx1-ubyte.gz": "25c81989df183df01b3e8a0aad5dffbe",
    "t10k-images-idx3-ubyte.gz": "bef4ecab320f06d8554ea6380940ec79",
    "t10k-labels-idx1-ubyte.gz": "bb300cfdad3c16e7a12a480ee83cd310",
}
CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal",
               "Shirt", "Sneaker", "Bag", "Ankle boot"]
OFFICIAL_ATTACK = {"epsilon": 0.10, "alpha": 0.01, "steps": 20,
                   "random_start": True, "loss": "cross_entropy", "norm": "linf",
                   "restarts": 1, "iterate": "final", "targeted": False}



import sys
sys.path.insert(0, str(ROOT/'src'))
assert (ROOT / 'data').exists(), 'Run the notebook from its project directory'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print({'device': str(DEVICE), 'torch': torch.__version__})
# False reviews the supplied records; True recomputes all confirmation experiments.
FRESH_TRAIN = False


{'device': 'cuda', 'torch': '2.11.0+cu128'}


In [2]:
def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2) + "\n", encoding="utf-8")


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.set_num_threads(4)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


def environment():
    return {"python": platform.python_version(), "torch": torch.__version__,
            "numpy": np.__version__, "platform": platform.platform(),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "cuda": torch.version.cuda, "precision": "float32", "tf32": False,
            "deterministic_algorithms": True, "cpu_threads": 4}


def sync(device):
    if str(device).startswith("cuda"):
        torch.cuda.synchronize(device)


def prepare_data():
    dest = ROOT / "data"
    dest.mkdir(parents=True, exist_ok=True)
    manifest = {}
    for name, md5 in FILES.items():
        path = dest / name
        source = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/" + name
        if not path.exists():
            print("Downloading", name, flush=True)
            urllib.request.urlretrieve(source, path)
        blob = path.read_bytes()
        if hashlib.md5(blob).hexdigest() != md5:
            raise ValueError(f"Dataset checksum mismatch: {name}")
        manifest[name] = {"source": source, "md5": md5,
                          "sha256": hashlib.sha256(blob).hexdigest(), "bytes": len(blob)}
    save_json(dest / "manifest.json", manifest)
    return manifest


def read_idx(path):
    blob = gzip.decompress(Path(path).read_bytes())
    magic, count = struct.unpack(">II", blob[:8])
    if magic == 2051:
        rows, cols = struct.unpack(">II", blob[8:16])
        return np.frombuffer(blob, dtype=np.uint8, offset=16).copy().reshape(count, rows * cols)
    if magic == 2049:
        return np.frombuffer(blob, dtype=np.uint8, offset=8).copy()
    raise ValueError("Unsupported IDX header")


def split_indices(labels):
    rng = np.random.default_rng(7404)
    train, valid = [], []
    for label in range(10):
        idx = np.flatnonzero(labels == label)
        rng.shuffle(idx)
        valid.extend(idx[:500])
        train.extend(idx[500:])
    train, valid = np.sort(train), np.sort(valid)
    assert len(train) == 55000 and len(valid) == 5000
    assert not np.intersect1d(train, valid).size
    np.savez(ROOT / "data" / "split_indices.npz", train=train, validation=valid)
    save_json(ROOT / "data" / "split_manifest.json", {
        "seed": 7404, "stratification": "500 validation examples per class",
        "training_examples": len(train), "validation_examples": len(valid),
        "train_indices_sha256": hashlib.sha256(train.tobytes()).hexdigest(),
        "validation_indices_sha256": hashlib.sha256(valid.tobytes()).hexdigest(),
        "preprocessing": "uint8 / 255, pixels in [0,1]; no augmentation or normalization"})
    return train, valid


def load_data(device, include_test=False):
    prepare_data()
    x = read_idx(ROOT / "data" / "train-images-idx3-ubyte.gz")
    y = read_idx(ROOT / "data" / "train-labels-idx1-ubyte.gz")
    train, valid = split_indices(y)
    out = {"train": (torch.tensor(x[train], device=device, dtype=torch.float32) / 255,
                     torch.tensor(y[train], device=device, dtype=torch.long)),
           "validation": (torch.tensor(x[valid], device=device, dtype=torch.float32) / 255,
                          torch.tensor(y[valid], device=device, dtype=torch.long))}
    if include_test:
        out["test"] = (torch.tensor(read_idx(ROOT / "data" / "t10k-images-idx3-ubyte.gz"),
                                    device=device, dtype=torch.float32) / 255,
                       torch.tensor(read_idx(ROOT / "data" / "t10k-labels-idx1-ubyte.gz"),
                                    device=device, dtype=torch.long))
    return out


def make_model(device):
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
                         nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)


In [3]:
seed_all(7411)
if FRESH_TRAIN:
    DATA = load_data(DEVICE)
    print('Current training environment:', environment())
    print({k: (tuple(x.shape), tuple(y.shape)) for k, (x, y) in DATA.items()})
else:
    DATA = None
    print('Recorded experiment environment:', json.loads((ROOT/'results/environment.json').read_text()))
    split = np.load(ROOT/'data/split_indices.npz')
    print({'training_examples': len(split['train']), 'validation_examples': len(split['validation']),
           'raw_images_loaded': False, 'mode': 'saved-record review'})
print(make_model(DEVICE))
assert sum(p.numel() for p in make_model(DEVICE).parameters()) == 235146


Recorded experiment environment: {'python': '3.12.14', 'torch': '2.11.0+cu128', 'numpy': '2.5.2', 'platform': 'Windows-11-10.0.26200-SP0', 'gpu': 'NVIDIA GeForce RTX 4060', 'cuda': '12.8', 'precision': 'float32', 'tf32': False, 'deterministic_algorithms': True, 'cpu_threads': 4, 'package_versions': {'torch': '2.11.0+cu128', 'numpy': '2.5.2', 'pandas': '3.0.5', 'matplotlib': '3.11.1', 'nbformat': '5.11.1', 'nbclient': '0.11.0', 'nbconvert': '7.17.1', 'ipykernel': '7.3.0', 'notebook': '7.6.2', 'reportlab': '4.4.9', 'pypdf': '6.10.0', 'pypdfium2': '5.13.0', 'Pillow': '12.3.0'}}
{'training_examples': 55000, 'validation_examples': 5000, 'raw_images_loaded': False, 'mode': 'saved-record review'}


Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=128, bias=True)
  (4): ReLU()
  (5): Linear(in_features=128, out_features=10, bias=True)
)


## 2. The official cross-entropy PGD attack

The attack is untargeted, uses raw pixel coordinates, and projects into both the pixel domain and the L-infinity ball. Official evaluation uses epsilon 0.10, step size 0.01, twenty steps, a random start and the final iterate. Gradients are taken with respect to the input only.

In [4]:
def margin(logits, labels):
    true = logits.gather(1, labels[:, None]).squeeze(1)
    other = logits.clone()
    other.scatter_(1, labels[:, None], -torch.inf)
    return true - other.amax(1)


def pgd(model, x, y, steps=20, epsilon=.10, alpha=.01, generator=None,
        start=None, keep_best=False):
    """Untargeted CE ascent in raw pixel coordinates; model gradients are untouched."""
    lower, upper = (x - epsilon).clamp(0, 1), (x + epsilon).clamp(0, 1)
    if start is None:
        noise = torch.rand(x.shape, device=x.device, generator=generator) * (2 * epsilon) - epsilon
        adv = (x + noise).clamp(0, 1).detach()
    else:
        adv = torch.maximum(torch.minimum(start.detach(), upper), lower)
    best = adv.clone() if keep_best else None
    best_loss = torch.full_like(y, -torch.inf, dtype=x.dtype) if keep_best else None
    for _ in range(steps):
        adv.requires_grad_(True)
        logits = model(adv)
        losses = F.cross_entropy(logits, y, reduction="none")
        if keep_best:
            improve = losses.detach() > best_loss
            best[improve] = adv.detach()[improve]
            best_loss = torch.maximum(best_loss, losses.detach())
        grad = torch.autograd.grad(losses.sum(), adv, only_inputs=True)[0]
        adv = adv.detach() + alpha * grad.sign()
        adv = torch.maximum(torch.minimum(adv, upper), lower).detach()
    if keep_best:
        with torch.no_grad():
            last_loss = F.cross_entropy(model(adv), y, reduction="none")
            improve = last_loss > best_loss
            best[improve] = adv[improve]
        return best.detach()
    return adv


def clean_predictions(model, x, batch=512):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(x[i:i+batch]).argmax(1) for i in range(0, len(x), batch)])


def evaluate(model, x, y, seed=9001, steps=20, alpha=.01, restarts=1,
             keep_best=False, batch=512):
    model.eval()
    clean = clean_predictions(model, x, batch)
    rng = torch.Generator(device=x.device).manual_seed(seed)
    adv_predictions, adv_loss = [], []
    for i in range(0, len(x), batch):
        xb, yb = x[i:i+batch], y[i:i+batch]
        chosen_loss = torch.full_like(yb, -torch.inf, dtype=xb.dtype)
        chosen_pred = torch.zeros_like(yb)
        any_failure = torch.zeros_like(yb, dtype=torch.bool)
        for _ in range(restarts):
            adv = pgd(model, xb, yb, steps, .10, alpha, rng, keep_best=keep_best)
            with torch.no_grad():
                logits = model(adv)
                loss = F.cross_entropy(logits, yb, reduction="none")
                pred = logits.argmax(1)
                # Preserve any successful restart; among failures retain the largest CE.
                failed = pred != yb
                choose = (failed & ~any_failure) | ((failed == any_failure) & (loss > chosen_loss))
                chosen_pred[choose] = pred[choose]
                chosen_loss[choose] = loss[choose]
                any_failure |= failed
        adv_predictions.append(chosen_pred)
        adv_loss.append(chosen_loss)
    pred = torch.cat(adv_predictions)
    losses = torch.cat(adv_loss)
    metrics = {"n": len(y), "clean_correct": int((clean == y).sum()),
               "robust_correct": int((pred == y).sum()),
               "clean_accuracy": float((clean == y).float().mean()),
               "pgd_accuracy": float((pred == y).float().mean()),
               "attack_seed": seed, "steps": steps, "alpha": alpha,
               "epsilon": .10, "restarts": restarts, "keep_best": keep_best}
    return metrics, {"label": y.cpu().numpy(), "clean_prediction": clean.cpu().numpy(),
                     "adversarial_prediction": pred.cpu().numpy(), "attack_loss": losses.cpu().numpy()}


def check_attack(device):
    seed_all(17)
    model = make_model(device)
    x = torch.rand((32, 784), device=device)
    y = torch.randint(0, 10, (32,), device=device)
    model.eval()
    params = [p.detach().clone() for p in model.parameters()]
    rng = torch.Generator(device=device).manual_seed(71)
    adv = pgd(model, x, y, generator=rng)
    assert adv.min() >= 0 and adv.max() <= 1
    assert float((adv-x).abs().max()) <= .100001
    assert all(p.grad is None for p in model.parameters())
    assert all(torch.equal(a, b) for a, b in zip(params, model.parameters()))
    again = pgd(model, x, y, generator=torch.Generator(device=device).manual_seed(71))
    assert torch.equal(adv, again)
    zero = pgd(model, x, y, epsilon=0, generator=rng)
    assert torch.equal(zero, x)
    # An independent implementation checks the official final-iterate PGD numerically.
    gen = torch.Generator(device=device).manual_seed(71)
    reference = (x + torch.rand(x.shape, device=device, generator=gen)*.2-.1).clamp(0, 1)
    for _ in range(20):
        reference = reference.detach().requires_grad_(True)
        gradient = torch.autograd.grad(F.cross_entropy(model(reference), y), reference)[0]
        reference = (reference.detach()+.01*gradient.sign()).clamp(x-.1, x+.1).clamp(0, 1)
    assert torch.allclose(reference, adv, atol=1e-6)
    assert sum(p.numel() for p in model.parameters()) == 235146
    return {"pixel_bounds": True, "linf_projection": True, "parameter_unchanged": True,
            "parameter_gradients_untouched": True, "repeatability": True, "epsilon_zero": True,
            "independent_pgd_agreement": True, "parameters": 235146,
            "max_perturbation": float((adv-x).abs().max())}


In [5]:
from types import SimpleNamespace
core = SimpleNamespace(**{name: globals()[name] for name in ['save_json','seed_all','environment','sync','prepare_data','read_idx','split_indices','load_data','make_model','margin','pgd','clean_predictions','evaluate','check_attack','OFFICIAL_ATTACK','CLASS_NAMES']})
print(check_attack(DEVICE))

{'pixel_bounds': True, 'linf_projection': True, 'parameter_unchanged': True, 'parameter_gradients_untouched': True, 'repeatability': True, 'epsilon_zero': True, 'independent_pgd_agreement': True, 'parameters': 235146, 'max_perturbation': 0.10000002384185791}


## 3. Loss, continuation and the final parameter state

The proposed outer objective is

$$\mathcal{L}=0.25\,\mathrm{CE}(f_\theta(x),y)+0.75\,\mathrm{CE}(f_\theta(x_{\rm adv}),y)+0.5D_{\rm KL}(p_\theta(x)\Vert p_\theta(x_{\rm adv})).$$

Both clean and adversarial output distributions retain their gradients in this KL term. The inner attack maximizes cross-entropy. The radius is $\epsilon_e=0.10\min(1,e/10)$, with five steps and $\alpha_e=\epsilon_e/4$. The inference model averages epoch-end parameters from epochs 21–30. These settings are selected on validation before the five confirmation seeds are tested. TRADES, MART and confidence-weighted consistency are retained as development alternatives in the same loss implementation.


In [6]:
"""Validation-only development of consistency objectives and parameter averaging."""
from __future__ import annotations

import argparse
import copy
import csv
import hashlib
import json
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import numpy as np
import torch
from torch.nn import functional as F


ROOT = Path.cwd()



@dataclass(frozen=True)
class Config:
    tag: str
    objective: str = "mixed"
    seed: int = 7404
    epochs: int = 30
    batch_size: int = 256
    lr: float = .001
    weight_decay: float = .0001
    steps: int = 5
    beta: float = 2.
    clean_weight: float = .25
    ramp_epochs: int = 0
    warmup_steps: int | None = None
    averaging: str = "none"
    ema_decay: float = .995
    average_start: int = 5
    stage: str = "development"
    trace_validation: bool = False

    @property
    def name(self):
        return f"{self.stage}_{self.tag}_s{self.seed}_e{self.epochs}"


In [7]:
def kl_divergence(clean_logits, adversarial_logits):
    """KL(p_clean || p_adv), retaining both outer-loss gradient paths."""
    p = F.softmax(clean_logits, dim=1)
    return (p * (F.log_softmax(clean_logits, dim=1) -
                 F.log_softmax(adversarial_logits, dim=1))).sum(1)


def kl_attack(model, x, steps, epsilon, alpha, generator):
    """The clean distribution is fixed during the inner maximization."""
    lower, upper = (x-epsilon).clamp(0, 1), (x+epsilon).clamp(0, 1)
    with torch.no_grad():
        p = F.softmax(model(x), dim=1).detach()
    noise = torch.rand(x.shape, device=x.device, generator=generator)*2*epsilon-epsilon
    adv = (x+noise).clamp(0, 1).detach()
    for _ in range(steps):
        adv.requires_grad_(True)
        objective = F.kl_div(F.log_softmax(model(adv), dim=1), p, reduction="sum")
        grad = torch.autograd.grad(objective, adv, only_inputs=True)[0]
        adv = torch.maximum(torch.minimum(adv.detach()+alpha*grad.sign(), upper), lower).detach()
    return adv


def objective(model, x, y, cfg, epsilon, rng):
    if cfg.objective == "standard":
        return F.cross_entropy(model(x), y)
    alpha = epsilon/4
    if cfg.objective == "trades":
        adv = kl_attack(model, x, cfg.steps, epsilon, alpha, rng)
        clean_logits, adv_logits = model(x), model(adv)
        return F.cross_entropy(clean_logits, y) + cfg.beta*kl_divergence(clean_logits, adv_logits).mean()
    adv = core.pgd(model, x, y, cfg.steps, epsilon, alpha, rng)
    adv_logits = model(adv)
    if cfg.objective == "pgd":
        return F.cross_entropy(adv_logits, y)
    clean_logits = model(x)
    if cfg.objective == "mixed":
        return cfg.clean_weight*F.cross_entropy(clean_logits, y) + (1-cfg.clean_weight)*F.cross_entropy(adv_logits, y)
    if cfg.objective in {"hybrid", "gated"}:
        penalty = kl_divergence(clean_logits, adv_logits)
        if cfg.objective == "gated":
            confidence = F.softmax(clean_logits, dim=1).gather(1, y[:, None]).squeeze(1).detach()
            penalty = confidence*penalty
        return (cfg.clean_weight*F.cross_entropy(clean_logits, y) +
                (1-cfg.clean_weight)*F.cross_entropy(adv_logits, y) +
                cfg.beta*penalty.mean())
    if cfg.objective == "mart":
        prob = F.softmax(adv_logits, dim=1)
        other = prob.detach().clone()
        other.scatter_(1, y[:, None], -torch.inf)
        rival = other.argmax(1)
        boosted = F.cross_entropy(adv_logits, y) - torch.log((1-prob.gather(1, rival[:, None]).squeeze(1)).clamp_min(1e-12)).mean()
        confidence = F.softmax(clean_logits, dim=1).gather(1, y[:, None]).squeeze(1)
        return boosted + cfg.beta*((1-confidence)*kl_divergence(clean_logits, adv_logits)).mean()
    raise ValueError(cfg.objective)


## 4. Optimization and synchronized timing

The complete loop below includes attack generation, both supervised branches, consistency gradients and parameter averaging. Validation traces and checkpoint serialization are outside the optimization timer. No early stopping or test-dependent checkpoint choice is used.

In [8]:
def train(cfg, data, device, force=False):
    folder = ROOT/"runs"/cfg.name
    checkpoint = ROOT/"checkpoints"/(cfg.name+".pt")
    average_checkpoint = ROOT/"checkpoints"/(cfg.name+"_average.pt")
    have_weights = checkpoint.exists() and (cfg.averaging == "none" or average_checkpoint.exists())
    if (folder/"training.json").exists() and (folder/"validation.json").exists() and have_weights and not force:
        record = json.loads((folder/"training.json").read_text())
        record["config"].setdefault("warmup_steps", None)
        if record["config"] != asdict(cfg):
            raise ValueError("Cached configuration differs")
        return record
    folder.mkdir(parents=True, exist_ok=True)
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    core.seed_all(cfg.seed)
    model = core.make_model(device)
    averaged = copy.deepcopy(model) if cfg.averaging != "none" else None
    optim = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, cfg.epochs, eta_min=1e-5)
    shuffle_rng = torch.Generator(device=device).manual_seed(cfg.seed+10000)
    attack_rng = torch.Generator(device=device).manual_seed(cfg.seed+20000)
    x, y = data["train"]
    vx, vy = data["validation"]
    trace_idx = torch.cat([torch.where(vy == k)[0][:100] for k in range(10)])
    optim.zero_grad(set_to_none=True)
    F.cross_entropy(model(x[:cfg.batch_size]), y[:cfg.batch_size]).backward()
    optim.zero_grad(set_to_none=True)
    core.sync(device)
    started = time.perf_counter()
    excluded = 0.
    history, trace = [], []
    n_average = 0
    for epoch in range(cfg.epochs):
        core.sync(device)
        epoch_started = time.perf_counter()
        model.train()
        permutation = torch.randperm(len(y), device=device, generator=shuffle_rng)
        epsilon = .10*min(1., (epoch+1)/cfg.ramp_epochs) if cfg.ramp_epochs else .10
        steps = cfg.warmup_steps if cfg.warmup_steps is not None and epoch < cfg.ramp_epochs else cfg.steps
        step_cfg = replace(cfg, steps=steps)
        loss_sum = torch.zeros((), device=device)
        lr = optim.param_groups[0]["lr"]
        for i in range(0, len(y), cfg.batch_size):
            idx = permutation[i:i+cfg.batch_size]
            xb, yb = x[idx], y[idx]
            optim.zero_grad(set_to_none=True)
            loss = objective(model, xb, yb, step_cfg, epsilon, attack_rng)
            loss.backward()
            optim.step()
            if averaged is not None and cfg.averaging == "ema":
                with torch.no_grad():
                    src, dst = list(model.parameters()), list(averaged.parameters())
                    if epoch < cfg.average_start:
                        torch._foreach_copy_(dst, src)
                    else:
                        torch._foreach_mul_(dst, cfg.ema_decay)
                        torch._foreach_add_(dst, src, alpha=1-cfg.ema_decay)
            loss_sum += loss.detach()*len(yb)
        if averaged is not None and cfg.averaging == "swa" and epoch >= cfg.average_start:
            with torch.no_grad():
                src, dst = list(model.parameters()), list(averaged.parameters())
                torch._foreach_mul_(dst, n_average/(n_average+1))
                torch._foreach_add_(dst, src, alpha=1/(n_average+1))
                n_average += 1
        scheduler.step()
        core.sync(device)
        elapsed = time.perf_counter()-epoch_started
        row = {"epoch": epoch+1, "loss": float(loss_sum/len(y)), "lr": lr,
               "epsilon": epsilon, "seconds": elapsed,
               "mean_attack_steps": 0 if cfg.objective == "standard" else steps}
        history.append(row)
        print(cfg.name, f"epoch {epoch+1:02d}: loss={row['loss']:.4f}, time={elapsed:.2f}s", flush=True)
        core.save_json(folder/"progress.json", row)
        if cfg.trace_validation and (epoch+1) % 5 == 0:
            core.sync(device)
            paused = time.perf_counter()
            for version, current in [("raw", model), ("average", averaged)]:
                if current is not None:
                    if version == "average" and cfg.averaging == "swa" and epoch < cfg.average_start:
                        continue  # The late arithmetic average is not defined yet.
                    scores, _ = core.evaluate(current, vx[trace_idx], vy[trace_idx], seed=19001)
                    trace.append({"epoch": epoch+1, "version": version, **scores})
            core.sync(device)
            excluded += time.perf_counter()-paused
    core.sync(device)
    total = time.perf_counter()-started-excluded
    torch.save(model.state_dict(), checkpoint)
    if averaged is not None:
        torch.save(averaged.state_dict(), average_checkpoint)
    record = {"config": asdict(cfg), "environment": core.environment(), "history": history,
              "training_seconds": total, "epochs": cfg.epochs,
              "mean_epoch_seconds": sum(h["seconds"] for h in history)/cfg.epochs,
              "diagnostic_validation_seconds": excluded,
              "mean_attack_steps": sum(h["mean_attack_steps"] for h in history)/cfg.epochs,
              "timing": "Synchronized wall time of all optimization, attack generation, parameter averaging, epoch logs and updates; excludes preparation, warmup, diagnostic validation and checkpoint serialization",
              "checkpoint_sha256": hashlib.sha256(checkpoint.read_bytes()).hexdigest()}
    core.save_json(folder/"training.json", record)
    with (folder/"history.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=history[0].keys())
        writer.writeheader()
        writer.writerows(history)
    if trace:
        core.save_json(folder/"validation_trace.json", trace)
    scores, arrays = core.evaluate(model, vx, vy, seed=19001)
    core.save_json(folder/"validation_raw.json", scores)
    np.savez_compressed(folder/"validation_predictions_raw.npz", **arrays)
    if averaged is not None:
        scores, arrays = core.evaluate(averaged, vx, vy, seed=19001)
        core.save_json(folder/"validation_average.json", scores)
        np.savez_compressed(folder/"validation_predictions_average.npz", **arrays)
    core.save_json(folder/"validation.json", scores)
    print(cfg.name, "validation", json.dumps(scores), flush=True)
    return record


## 5. Validation selection and the frozen confirmatory suite

The development ledger includes twenty-two thirty-epoch configurations. The clean floor is 83.74%; candidates within 0.20 percentage points of the highest eligible validation PGD-20 accuracy are compared by measured training time. The confirmation uses seeds 7411–7415. An earlier study had already evaluated test data; this revision uses validation for configuration choices and freezes the new seeds before their test evaluation.

In [9]:
"""Freeze validation selection, train paired seeds, and evaluate the frozen suite."""
from __future__ import annotations
import argparse
from dataclasses import asdict, replace
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd
import torch


SEEDS = [7411, 7412, 7413, 7414, 7415]
FAMILY = ["standard", "pgd", "mixed", "curriculum_mixed", "consistency", "proposed"]




def freeze():
    path = ROOT/"results/frozen_protocol.json"
    if path.exists():
        return json.loads(path.read_text())
    if not (ROOT/"results/budget_pilot.json").exists():
        raise RuntimeError("Complete the declared validation phases before freezing")
    candidates = []
    for folder in sorted((ROOT/"runs").glob("development_*")):
        record = json.loads((folder/"training.json").read_text())
        scores = json.loads((folder/"validation.json").read_text())
        cfg = dict(record["config"])
        cfg.setdefault("warmup_steps", None)
        candidates.append({"config": cfg, "run": folder.name,
                           "clean": scores["clean_correct"]/scores["n"],
                           "pgd20": scores["robust_correct"]/scores["n"],
                           "training_seconds": record["training_seconds"]})
    eligible = [c for c in candidates if c["clean"] >= .8374]
    best = max(c["pgd20"] for c in eligible)
    close = [c for c in eligible if c["pgd20"] >= best-.002-1e-12]
    chosen = min(close, key=lambda c: c["training_seconds"])
    proposed = Config(**chosen["config"])
    if proposed.objective != "hybrid" or proposed.averaging != "swa":
        raise RuntimeError("The confirmatory factorial design must match the selected family")
    base = replace(proposed, tag="proposed", seed=SEEDS[0], stage="confirmation", trace_validation=True)
    templates = [
        replace(base, tag="standard", objective="standard", ramp_epochs=0, warmup_steps=None, averaging="none"),
        replace(base, tag="pgd", objective="pgd", ramp_epochs=0, warmup_steps=None, averaging="none"),
        replace(base, tag="mixed", objective="mixed", ramp_epochs=0, warmup_steps=None),
        replace(base, tag="curriculum_mixed", objective="mixed"),
        replace(base, tag="consistency", ramp_epochs=0, warmup_steps=None),
        base,
    ]
    configs = []
    for k, seed in enumerate(SEEDS):
        # Rotate method order to spread elapsed-time drift over the comparisons.
        order = templates[k % len(templates):]+templates[:k % len(templates)]
        configs.extend(asdict(replace(c, seed=seed)) for c in order)
    protocol = {
        "frozen_at_utc": datetime.now(timezone.utc).isoformat(),
        "selection_rule": "Clean validation accuracy >= 83.74%; within 0.20 points of the highest validation PGD20 accuracy, choose the shortest measured optimization time",
        "selected_development": chosen, "all_development": candidates,
        "configs": configs, "seeds": SEEDS,
        "official_attack": core.OFFICIAL_ATTACK,
        "epochs": 30, "architecture_parameters": 235146,
        "selection_data": "55,000 training / 5,000 stratified validation; no new test evaluation",
        "revision_context": "An earlier completed study had evaluated the test set. New method selection uses validation only; the five confirmation seeds were not evaluated before this freeze.",
        "factorial_design": "Four optimization trajectories (mixed, curriculum_mixed, consistency, proposed) each provide final raw and averaged parameters: a 2x2x2 consistency/curriculum/averaging comparison. Standard and PGD baselines are additional trajectories.",
        "test_opened_before_freeze_in_this_revision": False,
    }
    core.save_json(path, protocol)
    core.save_json(ROOT/"results/environment.json", core.environment())
    print("Selected", chosen["run"], chosen["clean"], chosen["pgd20"], flush=True)
    return protocol


def configs():
    return [Config(**c) for c in freeze()["configs"]]


def train_suite(device, force=False):
    freeze()
    checks = core.check_attack(device)
    core.save_json(ROOT/"results/implementation_checks.json", checks)
    data = core.load_data(device)
    for cfg in configs():
        train(cfg, data, device, force=force)


def load_model(cfg, device, version="primary"):
    averaged = version == "average" or (version == "primary" and cfg.averaging != "none")
    suffix = "_average" if averaged else ""
    model = core.make_model(device)
    checkpoint = ROOT/"checkpoints"/(cfg.name+suffix+".pt")
    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
    model.eval()
    return model


def evaluate_suite(device, force=False):
    suite = configs()
    for cfg in suite:
        if not (ROOT/"runs"/cfg.name/"validation.json").exists():
            raise RuntimeError("Train all frozen configurations before reading test data")
    data = core.load_data(device, include_test=True)
    x, y = data["test"]
    rows = []
    for cfg in suite:
        record = json.loads((ROOT/"runs"/cfg.name/"training.json").read_text())
        versions = ["raw", "average"] if cfg.averaging != "none" else ["raw"]
        for version in versions:
            folder = ROOT/"runs"/cfg.name
            score_path = folder/(f"test_{version}.json")
            arrays_path = folder/(f"test_predictions_{version}.npz")
            if score_path.exists() and arrays_path.exists() and not force:
                score = json.loads(score_path.read_text())
            else:
                model = load_model(cfg, device, version)
                score, arrays = core.evaluate(model, x, y, seed=9001)
                core.save_json(score_path, score)
                np.savez_compressed(arrays_path, **arrays)
            rows.append({"run": cfg.name, "method": cfg.tag, "version": version,
                         "seed": cfg.seed, "epochs": cfg.epochs,
                         "training_seconds": record["training_seconds"],
                         "mean_epoch_seconds": record["mean_epoch_seconds"],
                         "mean_attack_steps": record["mean_attack_steps"],
                         "clean_accuracy": score["clean_correct"]/score["n"],
                         "pgd20_accuracy": score["robust_correct"]/score["n"],
                         "clean_correct": score["clean_correct"],
                         "robust_correct": score["robust_correct"], "n": score["n"]})
            print(cfg.name, version, score["clean_correct"], score["robust_correct"], flush=True)
    frame = pd.DataFrame(rows)
    frame.to_csv(ROOT/"results/all_results.csv", index=False)
    core.save_json(ROOT/"results/all_results.json", rows)
    core.save_json(ROOT/"results/evaluation_record.json", {
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "frozen_protocol_sha256": hashlib.sha256((ROOT/"results/frozen_protocol.json").read_bytes()).hexdigest(),
        "configuration_changed_after_test_evaluation": False,
        "models": len(rows), "independent_training_runs": len(suite), "test_examples_per_model": 10000})
    return frame


In [10]:
PROTOCOL = freeze()
print('Frozen at:', PROTOCOL['frozen_at_utc'])
print('Selected validation configuration:', PROTOCOL['selected_development'])
print('Confirmation seeds:', PROTOCOL['seeds'])
print('Distinct optimization trajectories:', len(configs()))

Frozen at: 2026-10-02T07:55:51.677435+00:00
Selected validation configuration: {'config': {'tag': 'hybrid_b05_ramp10_swa20', 'objective': 'hybrid', 'seed': 7404, 'epochs': 30, 'batch_size': 256, 'lr': 0.001, 'weight_decay': 0.0001, 'steps': 5, 'beta': 0.5, 'clean_weight': 0.25, 'ramp_epochs': 10, 'averaging': 'swa', 'ema_decay': 0.995, 'average_start': 20, 'stage': 'development', 'trace_validation': False, 'warmup_steps': None}, 'run': 'development_hybrid_b05_ramp10_swa20_s7404_e30', 'clean': 0.8418, 'pgd20': 0.6818, 'training_seconds': 48.819280399999116}
Confirmation seeds: [7411, 7412, 7413, 7414, 7415]
Distinct optimization trajectories: 30


In [11]:
from checks import run as run_checks
CHECKS = run_checks(DEVICE)
print(CHECKS)

{'pixel_bounds': True, 'linf_projection': True, 'parameter_unchanged': True, 'parameter_gradients_untouched': True, 'repeatability': True, 'epsilon_zero': True, 'independent_pgd_agreement': True, 'parameters': 235146, 'max_perturbation': 0.10000002384185791, 'kl_attack_projection': True, 'kl_attack_parameter_gradients_untouched': True, 'consistency_matches_independent_formula': True, 'consistency_identity': True, 'consistency_gradients_reach_both_branches': True, 'train_validation_disjoint': True, 'split_covers_original_training_only': True, 'arithmetic_average_matches_reference': True}


## 6. Train all baselines and component controls

The four mixed-loss families each produce a raw and averaged parameter state, yielding a full consistency × curriculum × averaging comparison. Their two states share optimization; they are not counted as separate training runs. Standard and pure PGD baselines complete thirty independent confirmation trajectories.

In [12]:
from review_records import review as review_saved_records
if FRESH_TRAIN:
    train_suite(DEVICE, force=True)
    print('Completed thirty fresh confirmation training trajectories.')
else:
    RECORD_CHECKS = review_saved_records()
    print(RECORD_CHECKS)


{'frozen_protocol_matches': True, 'saved_model_states_checked': 50, 'test_images_per_state': 10000, 'optimization_runs': 52, 'summary_statistics_match': True, 'test_counts_match_run_records': True, 'per_image_prediction_files_checked': 50, 'source_of_displayed_results': 'Saved experiment records; no fresh training in review mode'}


## 7. Full-test official evaluation

Every model is attacked separately using the identical required PGD-20 protocol. The test set does not choose a configuration or an epoch. Counts and example-level predictions are retained for all fifty raw/averaged model states.

In [13]:
if FRESH_TRAIN:
    TEST_RESULTS = evaluate_suite(DEVICE, force=True)
else:
    TEST_RESULTS = pd.read_csv(ROOT/'results/all_results.csv')
assert len(TEST_RESULTS) == 50
assert (TEST_RESULTS['n'] == 10000).all()
display(TEST_RESULTS[['method','version','seed','training_seconds','clean_accuracy','pgd20_accuracy']])


,method,version,seed,training_seconds,clean_accuracy,pgd20_accuracy
0,standard,raw,7411,14.410003,0.8952,0.0078
1,pgd,raw,7411,51.219398,0.8213,0.6693
2,mixed,raw,7411,53.508403,0.8363,0.6744
3,mixed,average,7411,53.508403,0.8352,0.6743
4,curriculum_mixed,raw,7411,49.383256,0.8414,0.6726
5,curriculum_mixed,average,7411,49.383256,0.8407,0.6715
6,consistency,raw,7411,49.258172,0.8269,0.6762
7,consistency,average,7411,49.258172,0.8272,0.6760
8,proposed,raw,7411,45.124720,0.8326,0.6788
9,proposed,average,7411,45.124720,0.8324,0.6792


## 8. Stronger attack checks

The additional audit retains failure at any intermediate iterate and combines the official attack with fifty-step CE and logit-margin attacks, each with three restarts. It is an empirical robustness check. The epsilon sweep uses one CE-PGD-50 restart at each nonzero radius.

In [14]:
from attack_audit import run as audit_suite
if FRESH_TRAIN:
    AUDIT_ROWS = audit_suite(DEVICE, force=True)
else:
    AUDIT_ROWS = json.loads((ROOT/'results/attack_audit.json').read_text())['rows']
display(pd.DataFrame(AUDIT_ROWS))


,method,seed,n,official_pgd20,ce50_3restart,margin50_3restart,combined,combined_correct,additional_failures,max_linf_ce,max_linf_margin
0,standard,7411,10000,0.0078,0.0061,0.0076,0.0059,59,19,0.1,0.1
1,pgd,7411,10000,0.6693,0.6621,0.6605,0.6524,6524,169,0.1,0.1
2,proposed,7411,10000,0.6792,0.6715,0.6719,0.6629,6629,163,0.1,0.1
3,pgd,7412,10000,0.6689,0.6620,0.6599,0.6508,6508,181,0.1,0.1
4,proposed,7412,10000,0.6805,0.6739,0.6740,0.6646,6646,159,0.1,0.1
5,standard,7412,10000,0.0093,0.0067,0.0089,0.0064,64,29,0.1,0.1
6,proposed,7413,10000,0.6818,0.6748,0.6758,0.6664,6664,154,0.1,0.1
7,standard,7413,10000,0.0167,0.0134,0.0190,0.0129,129,38,0.1,0.1
8,pgd,7413,10000,0.6704,0.6626,0.6628,0.6532,6532,172,0.1,0.1
9,proposed,7414,10000,0.6818,0.6744,0.6756,0.6651,6651,167,0.1,0.1


## 9. Main comparison and complete component analysis

All tables use means and sample standard deviations over the five matched training seeds. Example counts are recomputed from the saved predictions before the figures are produced.

In [15]:
from analyze_results import run as analyze_suite
if FRESH_TRAIN:
    SUMMARY = analyze_suite()
else:
    SUMMARY = json.loads((ROOT/'results/summary.json').read_text())
SUMMARY_FRAME = pd.DataFrame(SUMMARY)
display(SUMMARY_FRAME[['method','version','clean_accuracy_mean','clean_accuracy_sd',
                       'pgd20_accuracy_mean','pgd20_accuracy_sd','training_seconds_mean','training_seconds_sd']])


,method,version,clean_accuracy_mean,clean_accuracy_sd,pgd20_accuracy_mean,pgd20_accuracy_sd,training_seconds_mean,training_seconds_sd
0,standard,raw,0.89454,0.001258,0.01102,0.003870,10.670492,2.098539
1,pgd,raw,0.82034,0.001350,0.66966,0.001960,41.414672,5.484250
2,mixed,raw,0.83656,0.001762,0.67488,0.002581,45.709193,4.391736
3,mixed,average,0.83580,0.001991,0.67432,0.002425,45.709193,4.391736
4,curriculum_mixed,raw,0.84110,0.000592,0.67382,0.000968,44.765665,2.593753
5,curriculum_mixed,average,0.84052,0.000920,0.67276,0.001923,44.765665,2.593753
6,consistency,raw,0.82686,0.001629,0.67774,0.003053,46.509418,1.583064
7,consistency,average,0.82698,0.001636,0.67658,0.003613,46.509418,1.583064
8,proposed,raw,0.83304,0.000643,0.68170,0.001875,45.709582,0.405452
9,proposed,average,0.83262,0.001331,0.68062,0.001171,45.709582,0.405452


In [16]:
from IPython.display import Image as DisplayImage, Markdown
def display_figure(name):
    figure = ROOT/'figures'/(name+'.png')
    assert figure.exists()
    if FRESH_TRAIN:
        display(DisplayImage(filename=str(figure)))
    else:
        display(Markdown(f'![{name}](figures/{name}.png)'))
display_figure('main_tradeoff')
display_figure('validation_dynamics')


![main_tradeoff](figures/main_tradeoff.png)

![validation_dynamics](figures/validation_dynamics.png)

## 10. Class-dependent failures, paired gains and regressions

The images are chosen deterministically: the first seed-7411 index in each declared outcome category. Displayed attacks reproduce the original evaluation generator stream and are checked against stored predictions.

In [17]:
PER_CLASS = pd.read_csv(ROOT/'results/per_class.csv').query("method in ['pgd','proposed']")
display(PER_CLASS.groupby(['method','version','class'])[['clean_accuracy','robust_accuracy']].agg(['mean','std']))
print(json.loads((ROOT/'results/paired_comparisons.json').read_text()))
display_figure('class_robustness')
display_figure('class_confusion')
display_figure('failure_examples')
display_figure('epsilon_sweep')


clean_accuracy           robust_accuracy  \
                                       mean       std            mean   
method   version class                                                  
pgd      raw     Ankle boot          0.9354  0.002608          0.8630   
                 Bag                 0.9562  0.003271          0.8662   
                 Coat                0.7980  0.004637          0.5542   
                 Dress               0.9108  0.001789          0.7078   
                 Pullover            0.6632  0.007887          0.4820   
                 Sandal              0.9020  0.004472          0.7714   
                 Shirt               0.3230  0.011091          0.1156   
                 Sneaker             0.8914  0.005413          0.7452   
                 T-shirt/top         0.8608  0.003564          0.6818   
                 Trouser             0.9626  0.001673          0.9094   
proposed average Ankle boot          0.9344  0.002608          0.8644   
                 Bag                 0.9648  0.002490          0.8828   
                 Coat                0.7938  0.002683          0.5482   
                 Dress               0.9104  0.001673          0.7126   
                 Pullover            0.6962  0.004438          0.4814   
                 Sandal              0.9116  0.004159          0.7754   
                 Shirt               0.3994  0.012798          0.2042   
                 Sneaker             0.8942  0.001924          0.7440   
                 T-shirt/top         0.8574  0.003647          0.6754   
                 Trouser             0.9640  0.001225          0.9178   
         raw     Ankle boot          0.9336  0.002191          0.8634   
                 Bag                 0.9644  0.002074          0.8804   
                 Coat                0.7858  0.004438          0.5400   
                 Dress               0.9104  0.002966          0.7148   
                 Pullover            0.6990  0.006285          0.4838   
                 Sandal              0.9134  0.003912          0.7788   
                 Shirt               0.4078  0.007563          0.2214   
                 Sneaker             0.8930  0.002646          0.7412   
                 T-shirt/top         0.8588  0.005070          0.6756   
                 Trouser             0.9642  0.001095          0.9176   

                                        
                                   std  
method   version class                  
pgd      raw     Ankle boot   0.004848  
                 Bag          0.008497  
                 Coat         0.018633  
                 Dress        0.006419  
                 Pullover     0.005292  
                 Sandal       0.008764  
                 Shirt        0.015453  
                 Sneaker      0.005310  
                 T-shirt/top  0.007155  
                 Trouser      0.002881  
proposed average Ankle boot   0.004159  
                 Bag          0.003194  
                 Coat         0.009731  
                 Dress        0.005030  
                 Pullover     0.008019  
                 Sandal       0.003286  
                 Shirt        0.012029  
                 Sneaker      0.004472  
                 T-shirt/top  0.003209  
                 Trouser      0.001643  
         raw     Ankle boot   0.003847  
                 Bag          0.005030  
                 Coat         0.010368  
                 Dress        0.006380  
                 Pullover     0.009497  
                 Sandal       0.004087  
                 Shirt        0.008444  
                 Sneaker      0.003421  
                 T-shirt/top  0.005320  
                 Trouser      0.001342

{'rows': [{'seed': 7411, 'gained': 267, 'lost': 168, 'both_correct': 6525, 'both_wrong': 3040, 'robust_difference_pp': 0.990000000000002, 'clean_difference_pp': 1.1099999999999999}, {'seed': 7412, 'gained': 273, 'lost': 157, 'both_correct': 6532, 'both_wrong': 3038, 'robust_difference_pp': 1.1599999999999944, 'clean_difference_pp': 1.100000000000001}, {'seed': 7413, 'gained': 276, 'lost': 162, 'both_correct': 6542, 'both_wrong': 3020, 'robust_difference_pp': 1.1399999999999966, 'clean_difference_pp': 1.5600000000000058}, {'seed': 7414, 'gained': 280, 'lost': 187, 'both_correct': 6538, 'both_wrong': 2995, 'robust_difference_pp': 0.9299999999999975, 'clean_difference_pp': 1.3299999999999979}, {'seed': 7415, 'gained': 285, 'lost': 159, 'both_correct': 6513, 'both_wrong': 3043, 'robust_difference_pp': 1.2599999999999945, 'clean_difference_pp': 1.0399999999999965}], 'intervals': {'robust_difference_pp': {'mean': 1.095999999999997, 'sd': 0.1339029499301617, 'paired_t95_lower': 0.929737527830

![class_robustness](figures/class_robustness.png)

![class_confusion](figures/class_confusion.png)

![failure_examples](figures/failure_examples.png)

![epsilon_sweep](figures/epsilon_sweep.png)

## 11. Development record and independent reproduction

The full candidate ledger preserves failures as well as the selected configuration. The separate reproduction record trains the proposal from scratch in a fresh directory and compares both parameter states and all official test predictions.

In [18]:
display(pd.read_csv(ROOT/'results/development_candidates.csv'))
print('Optimization workload:', json.loads((ROOT/'results/workload.json').read_text()))
print('Fresh reproduction:', json.loads((ROOT/'results/reproduction_check.json').read_text()))
print('Analysis verification:', json.loads((ROOT/'results/analysis_checks.json').read_text()))

,run,clean,pgd20,training_seconds
0,development_gated_b05_ramp10_swa20_s7404_e30,0.8480,0.6766,48.520206
1,development_gated_b05_ramp10_warm2_swa20_s7404...,0.8440,0.6732,66.736521
2,development_gated_b1_ramp10_swa20_s7404_e30,0.8398,0.6808,70.153119
3,development_gated_b1_ramp10_warm2_swa20_s7404_e30,0.8410,0.6742,63.907725
4,development_hybrid_b025_ramp10_s7404_e30,0.8480,0.6800,53.736666
5,development_hybrid_b025_ramp10_swa20_s7404_e30,0.8456,0.6780,50.101914
6,development_hybrid_b025_s7404_e30,0.8408,0.6752,60.465588
7,development_hybrid_b025_swa20_s7404_e30,0.8400,0.6742,53.829278
8,development_hybrid_b05_ramp10_swa20_s7404_e30,0.8418,0.6818,48.819280
9,development_hybrid_b05_ramp10_warm2_swa20_s740...,0.8416,0.6750,66.681939


Optimization workload: {'development_runs': 22, 'confirmation_runs': 30, 'distinct_optimization_runs': 52, 'total_optimization_epochs': 1560, 'development_training_seconds': 1152.5791602999998, 'confirmation_training_seconds': 1173.895109799967, 'validation_trace_seconds': 12.227657900024496, 'raw_and_average_are_shared_optimization_trajectories': True}
Fresh reproduction: {'reference_run': 'confirmation_proposed_s7411_e30', 'fresh_run': 'reproduction_proposed_s7411_e30', 'fresh_directory': 'D:\\Assignment\\Coursework1_MC647418_revised\\tmp\\independent_reproduction', 'seed': 7411, 'epochs': 30, 'from_scratch': True, 'used_cached_weights': False, 'training_seconds': 58.3104332000039, 'versions': {'raw': {'parameter_tensors_bitwise_equal': True, 'tensors': 6, 'full_test_arrays_bitwise_equal': {'label': True, 'clean_prediction': True, 'adversarial_prediction': True, 'attack_loss': True}, 'test_metrics': {'n': 10000, 'clean_correct': 8326, 'robust_correct': 6788, 'clean_accuracy': 0.83259

## 12. Reproduction commands

The first code cell controls the notebook mode. `FRESH_TRAIN = False` reviews recorded measurements;
`True` downloads the original data, trains from scratch and repeats official and supplementary attacks.

From this project directory:

```text
python src/review_records.py
python src/run_suite.py check
python src/run_suite.py train --force
python src/run_suite.py evaluate --force
python src/run_suite.py audit --force
python src/run_suite.py analyze
```

The compact package omits raw dataset images, model weights and large per-image prediction archives.
These are regenerated by the training/evaluation commands. The GitHub-ready version additionally supplies
all fifty official per-image prediction archives. See `README.md` for setup and the complete file map.
